In [2]:
import numpy as np
from scipy.optimize import curve_fit
import math
from matplotlib import pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.ticker import FormatStrFormatter
import random
def func2(x,yc,alpha2,alpha4,alpha1,alpha3,beta1,w,a,c):
    
    u0=(x[1]-w)
    u1=1
    f0=alpha2*(u0*x[0]**(1/a))**2+alpha4*(u0*x[0]**(1/a))**4+alpha1*(u0*x[0]**(1/a))**1+alpha3*(u0*x[0]**(1/a))**3
    f1=beta1*(u1*x[0]**(-c))
    y=yc+f0+f1
    return y
def xi(x,yc,alpha2,alpha4,alpha1,alpha3,beta1,w,a,c):
    u0=abs(x[1]-w)
    return u0**(-a)
def corrected(x,yc,alpha2,alpha4,alpha1,alpha3,beta1,w,a,c):
    u0=(x[1]-w)
    u1=1
    f0=alpha2*(u0*x[0]**(1/a))**2+alpha4*(u0*x[0]**(1/a))**4+alpha1*(u0*x[0]**(1/a))**1+alpha3*(u0*x[0]**(1/a))**3
    f1=beta1*(u1*x[0]**(-c))
    y=yc+f0
    return y

%matplotlib inline
%config InlineBackend.figure_format = 'pdf'
series_low=-100000
series_high=100000
a_low=1
a_high=3
c_low=0
c_high=10
w_low=3.
w_high=3.25
yc_low=0
yc_high=10
bounds=([series_low,series_low,series_low,series_low,series_low,series_low,w_low,a_low,c_low],
        [series_high,series_high,series_high,series_high,series_high,series_high,w_high,a_high,c_high])
xx=np.array([
    [20 for i in range(1,20)]+[24 for i in range(1,20)]+[28 for i in range(1,20)]+[32 for i in range(1,20)]+[36 for i in range(1,20)]+[40 for i in range(1,20)]+[44 for i in range(1,20)]+[48 for i in range(1,20)],
    [2.95+0.02*i for i in range(1,20)]*8
])
zz=np.array([1.36712,1.29559,1.23855,1.1821,1.13439,1.10114,1.06744,1.05184,1.03709,1.03047,1.03781,1.04637,1.06438,1.08891,1.12415,1.16334,1.21779,1.2683,1.33676,
             1.41168,1.31924,1.24194,1.18115,1.12274,1.07622,1.05057,1.02969,1.01967,1.01387,1.01731,1.02728,1.04909,1.07591,1.12396,1.16841,1.22399,1.29084,1.3647,
             1.46164,1.34915,1.27328,1.19071,1.14092,1.09322,1.05271,1.01528,1.0052,0.99821,1.00455,1.0148,1.03959,1.08798,1.11321,1.1715,1.24257,1.31273,1.39451,
             1.49819,1.39131,1.29746,1.20717,1.1439,1.08989,1.04663,1.01566,0.99704,0.97801,0.99558,1.00215,1.03853,1.06855,1.13103,1.17613,1.26108,1.3232,1.42849,
             1.55227,1.413,1.31957,1.22376,1.15655,1.08521,1.04023,1.00578,0.98136,0.97644,0.98562,1,1.0312,1.08124,1.13391,1.19703,1.28013,1.36112,1.46803,
             1.59192,1.46306,1.34607,1.25288,1.15692,1.09461,1.03998,1.01452,0.98484,0.97508,0.97832,1.00497,1.03685,1.0772,1.13244,1.20891,1.29771,1.39658,1.50484,
            1.65161,1.50585,1.37689,1.27256,1.17797,1.09797,1.05202,1.00654,0.98146,0.97063,0.97154,0.98099,1.02258,1.06916,1.14895,1.22783,1.30714,1.42701,1.55101,
           1.70062,1.52803,1.39295,1.27647,1.1781,1.10631,1.04006,0.99668,0.9697,0.9691,0.9655,0.9865,1.0332,1.0895,1.1506,1.23488,1.34339,1.45544,1.60714
])
z=np.array([zz[i] for i in range(19*8)])
sigma_z=np.array([math.sqrt(xx[0][i])*z[i]/1000/math.sqrt(z[i])*1/math.sqrt(1.5) for i in range(19*8)])
initial_guess=[1,0,0,0,0,0,3.2,2.8,0.004]
abc,para=curve_fit(func2,xx,z,sigma=sigma_z,absolute_sigma=True,p0=initial_guess,bounds=bounds,maxfev=100000)
print(abc)
print(para)
z_fit=np.array([func2((xx[0][i],xx[1][i]),*abc) for i in range(19*8)])
chi=0
dev=[]
for i in range(19*8):
    chi+=(z[i]-z_fit[i])**2/sigma_z[i]**2
    dev.append((z[i]-z_fit[i])**2/sigma_z[i]**2)
print(dev)
print(chi)
amplitudes=np.linspace(20,48,8)
norm=Normalize(vmin=20,vmax=48)
cmap=plt.get_cmap('viridis')
sm=ScalarMappable(norm=norm,cmap=cmap)
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,4),sharey=True)
for i in range(8):
    ax1.errorbar(xx[1][i*19:i*19+19],z[i*19:i*19+19],fmt='o',yerr=sigma_z[i*19:i*19+19],color=sm.to_rgba(20+i*4),markersize=3,capsize=3,label=r'$L_y=$'+f'{(20+i*4):.0f}')
    x_fit=np.linspace(2.97,3.33,100)
    z_fit=[func2((xx[0][i*19],x_fit[j]),*abc) for j in range(100)]
    ax1.plot(x_fit,z_fit,color=sm.to_rgba(20+i*4))
ax1.set_xlabel(r'$1/B$',fontdict={'family':'Times New Roman','size':24})
ax1.set_ylabel(r'$\Gamma$',fontdict={'family':'Times New Roman','size':24})
ax1.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax1.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax1.set_xticks([3.00, 3.15, 3.30])
ax1.set_yticks([1.0, 1.25,1.5])
ax1.tick_params(axis='both', which='major', labelsize=16)
norml=[xx[0][i]/xi((xx[0][i],xx[1][i]),*abc) for i in range(19*8)]
gamma_c=[corrected((xx[0][i],xx[1][i]),*abc) for i in range(19*8)]
for i in range(8):
    ax2.errorbar(norml[i*19:i*19+19],z[i*19:i*19+19],fmt='o',yerr=sigma_z[i*19:i*19+19],color=sm.to_rgba(20+i*4),markersize=3,capsize=3)
ax2.set_xlabel(r'$L_y / \xi$',fontdict={'family':'Times New Roman','size':24})
ax2.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax1.text(0.02, 0.98, '(a)', transform=ax1.transAxes, fontsize=20, va='top')
ax2.text(0.02, 0.98, '(b)', transform=ax2.transAxes, fontsize=20, va='top')
ax2.text(0.7, 0.2, r'$L_z=1$', transform=ax2.transAxes, fontsize=20, va='top',bbox=dict(boxstyle='round', facecolor='white', edgecolor='silver'))
ax2.set_xticks([0, 0.3, 0.6])
ax2.tick_params(axis='both', which='major', labelsize=16)
plt.tight_layout()
fig.legend(loc='lower left',bbox_to_anchor=(0.19,0.555),fontsize=12,ncol=2)
plt.subplots_adjust(wspace=0)

[ 0.92479318  0.77908546  0.03788124  0.01597652 -0.04630703  3.3369534
  3.15434728  2.38146375  1.14681654]
[[ 9.60552660e-05  1.14559084e-04  3.10335193e-05  5.67952290e-06
  -9.98639706e-06  1.17348612e-02  6.88531409e-07  1.31942201e-04
   1.46692612e-03]
 [ 1.14559084e-04  7.83909763e-04  2.07442418e-04  3.93646392e-05
  -6.45223000e-05  9.20801481e-03  4.84099034e-06  8.80521367e-04
   1.31607676e-03]
 [ 3.10335193e-05  2.07442418e-04  1.82741973e-04  1.10610975e-05
  -1.45503097e-05  1.78048173e-03  1.63564842e-06  3.07316688e-04
   2.72318382e-04]
 [ 5.67952290e-06  3.93646392e-05  1.10610975e-05  6.04316430e-05
   1.01432488e-05  3.95715975e-04  9.25710259e-06  4.97632727e-05
   5.70577110e-05]
 [-9.98639706e-06 -6.45223000e-05 -1.45503097e-05  1.01432488e-05
   3.39615634e-05 -8.15926873e-04  3.09922083e-06 -7.06260611e-05
  -1.16359226e-04]
 [ 1.17348612e-02  9.20801481e-03  1.78048173e-03  3.95715975e-04
  -8.15926873e-04  1.53778223e+00  4.47013741e-05  1.01052314e-02
   

<Figure size 1000x400 with 2 Axes>

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import math
from matplotlib import pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.ticker import FormatStrFormatter
import random
def func(x,b1,b2,c0,a00,a11,a20,a21,w,a,c):
    phi1=(x[0]**(1.0/a))*(b1*(x[1]-w)+b2*(x[1]-w)**2)
    phi2=(c0)*(x[0]**(-c))
    y=a00+phi2+phi1*(1+a11*phi2)+phi1**2*(a20+a21*phi2)
    return y
def func2(x,yc,alpha1,alpha3,alpha2,alpha4,beta1,w,a,c):
    
    u0=(x[1]-w)
    u1=1
    f0=alpha2*(u0*x[0]**(1/a))**2+alpha4*(u0*x[0]**(1/a))**4+alpha1*(u0*x[0]**(1/a))**1+alpha3*(u0*x[0]**(1/a))**3
    f1=beta1*(u1*x[0]**(-c))
    y=yc+f0+f1
    return y
def xi(x,yc,alpha1,alpha3,alpha2,alpha4,beta1,w,a,c):
    u0=abs(x[1]-w)
    return u0**(-a)
%matplotlib inline
%config InlineBackend.figure_format = 'pdf'
series_low=-1000000
series_high=1000000
a_low=1
a_high=3
c_low=0
c_high=1.4
w_low=0.17
w_high=0.19
yc_low=0.3
yc_high=10
bounds=([series_low,series_low,series_low,series_low,series_low,series_low,w_low,a_low,c_low],
        [series_high,series_high,series_high,series_high,series_high,series_high,w_high,a_high,c_high])


xx=np.array([
    [20 for i in range(19)]+[24 for i in range(19)]+[28 for i in range(19)]+[32 for i in range(19)]+[36 for i in range(19)]
    +[40 for i in range(19)]+[44 for i in range(19)]+[48 for i in range(19)],
    [0.165+0.00125*i for i in range(19)]*8
])
zz=np.array([2.35442,2.21947,2.0944,2.00084,1.90635,1.83596,1.7803,1.73953,1.71719,1.71703,1.73461,1.76323,1.80806,1.86883,1.94855,2.03954,2.15665,2.29377,2.44242,
             2.44109,2.27868,2.13304,2.00708,1.91381,1.82656,1.77558,1.72759,1.69676,1.69584,1.71961,1.74538,1.80171,1.87241,1.97078,2.07476,2.21653,2.36539,2.54873,
             2.51595,2.33628,2.17675,2.04357,1.9349,1.8442,1.76478,1.71581,1.69372,1.68235,1.69891,1.73545,1.79893,1.88474,1.9828,2.1129,2.27601,2.45343,2.64957,
             2.60692,2.39606,2.20152,2.06362,1.94102,1.84076,1.75682,1.70364,1.67296,1.66506,1.6793,1.73651,1.79554,1.88624,2.00685,2.15383,2.32632,2.53956,2.75855,
             2.70876,2.46285,2.25943,2.08268,1.94456,1.82624,1.74079,1.68605,1.64652,1.66171,1.66478,1.72319,1.81341,1.89176,2.03698,2.20292,2.38184,2.6159,2.86283,
             2.77388,2.52695,2.30345,2.12464,1.95888,1.84865,1.75717,1.68542,1.64475,1.64292,1.66256,1.70642,1.80864,1.91294,2.06123,2.23721,2.44664,2.67978,2.96993,
             2.86763,2.60767,2.35899,2.15789,1.991,1.83603,1.74142,1.67435,1.63979,1.63135,1.6514,1.71166,1.80924,1.94146,2.08159,2.2829,2.50453,2.7761,3.07196,
            2.96971,2.66433,2.40997,2.17278,2.00016,1.86214,1.74624,1.66967,1.6232,1.63157,1.64526,1.71769,1.80782,1.94127,2.11824,2.32602,2.57466,2.85402,3.19148
])
z=np.array([zz[i] for i in range(19*8)])
sigma_z=np.array([math.sqrt(xx[0][i])*z[i]/1000/math.sqrt(z[i])*1/math.sqrt(1.5) for i in range(19*8)])
initial_guess=[1,0,0,0,0,0,0.18,2.3,0.4]
abc,para=curve_fit(func2,xx,z,sigma=sigma_z,absolute_sigma=True,p0=initial_guess,bounds=bounds,maxfev=100000)
print(abc)
print(para)
z_fit=np.array([func2((xx[0][i],xx[1][i]),*abc) for i in range(19*8)])
chi=0
dev=[]
for i in range(19*8):
    chi+=(z[i]-z_fit[i])**2/sigma_z[i]**2
    dev.append((z[i]-z_fit[i])**2/sigma_z[i]**2)
print(dev)
print(chi)
amplitudes=np.linspace(20,48,8)
norm=Normalize(vmin=20,vmax=48)
cmap=plt.get_cmap('viridis')
sm=ScalarMappable(norm=norm,cmap=cmap)
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,4),sharey=True)
for i in range(8):
    ax1.errorbar(xx[1][i*19:i*19+19],z[i*19:i*19+19],fmt='o',yerr=sigma_z[i*19:i*19+19],color=sm.to_rgba(20+i*4),markersize=3,capsize=3,label=r'$L_y=$'+f'{(20+i*4):.0f}')
    x_fit=np.linspace(0.165,0.1875,100)
    z_fit=[func2((xx[0][i*21],x_fit[j]),*abc) for j in range(100)]
    ax1.plot(x_fit,z_fit,color=sm.to_rgba(20+i*4))
ax1.set_xlabel(r'$E$',fontdict={'family':'Times New Roman','size':24})
ax1.set_ylabel(r'$\Gamma$',fontdict={'family':'Times New Roman','size':24})
ax1.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
ax1.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax1.set_xticks([0.165, 0.175, 0.185])
ax1.set_yticks([2.0, 3.0, 4.0])
ax1.tick_params(axis='both', which='major', labelsize=16)
norml=[xx[0][i]/xi((xx[0][i],xx[1][i]),*abc) for i in range(19*8)]
for i in range(8):
    ax2.errorbar(norml[i*19:i*19+19],z[i*19:i*19+19],fmt='o',yerr=sigma_z[i*19:i*19+19],color=sm.to_rgba(20+i*4),markersize=3,capsize=3)
ax2.set_xlabel(r'$L_y / \xi$',fontdict={'family':'Times New Roman','size':24})
ax2.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
ax1.text(0.02, 0.98, '(a)', transform=ax1.transAxes, fontsize=20, va='top')
ax2.text(0.02, 0.98, '(b)', transform=ax2.transAxes, fontsize=20, va='top')
ax2.text(0.7, 0.2, r'$L_z=7$', transform=ax2.transAxes, fontsize=20, va='top',bbox=dict(boxstyle='round', facecolor='white', edgecolor='silver'))
ax2.set_xticks([0, 0.001])
ax2.tick_params(axis='both', which='major', labelsize=16)
plt.tight_layout()
fig.legend(loc='lower left',bbox_to_anchor=(0.175,0.555),fontsize=12,ncol=2)
plt.subplots_adjust(wspace=0)

In [3]:
import numpy as np
from scipy.optimize import curve_fit
import math
from matplotlib import pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.ticker import FormatStrFormatter
def func2(x,yc,alpha1,alpha2,alpha3,alpha4,beta1,w,a,c):
    u0=(x[1]-w)
    u1=1
    f0=alpha2*(u0*x[0]**(1/a))**2+alpha4*(u0*x[0]**(1/a))**4+alpha1*(u0*x[0]**(1/a))**1+alpha3*(u0*x[0]**(1/a))**3
    f1=beta1*(u1*x[0]**(-c))
    y=yc+f0+f1
    return y
def xi(x,yc,alpha1,alpha2,alpha3,alpha4,beta1,w,a,c):
    u0=abs(x[1]-w)
    return u0**(-a)
def corrected(x,yc,alpha1,alpha2,alpha3,alpha4,beta1,w,a,c):
    u0=(x[1]-w)
    u1=1
    f0=alpha2*(u0*x[0]**(1/a))**2+alpha4*(u0*x[0]**(1/a))**4+alpha1*(u0*x[0]**(1/a))**1+alpha3*(u0*x[0]**(1/a))**3
    f1=beta1*(u1*x[0]**(-c))
    y=yc+f0
    return y

%matplotlib inline
%config InlineBackend.figure_format = 'pdf'
series_low=-500000
series_high=500000
a_low=2
a_high=3
c_low=0
c_high=10
w_low=0.19
w_high=0.22
y_low=-2
y_high=2
bounds=([series_low,series_low,series_low,series_low,series_low,series_low,w_low,a_low,c_low],
        [series_high,series_high,series_high,series_high,series_high,series_high,w_high,a_high,c_high])


xx=np.array([[20 for i in range(21)]+[24 for i in range(21)]+[28 for i in range(21)]+[32 for i in range(21)]+[36 for i in range(21)]+[40 for i in range(21)]+[44 for i in range(21)]+[48 for i in range(21)],
             
             [0.19+0.00125*i for i in range(21)]*8])
zz=np.array([2.44723,2.28622,2.14796,2.01392,1.91382,1.83501,1.75813,1.7105,1.68157,1.66579,1.67003,1.68776,1.72376,1.78074,1.86605,1.95381,2.06113,2.17948,2.32496,2.46848,2.63107,
             2.56289,2.36884,2.1923,2.0457,1.92729,1.83053,1.75682,1.68721,1.66233,1.64324,1.65344,1.68101,1.72279,1.79389,1.8787,1.98291,2.11544,2.26943,2.42804,2.60473,2.80632,
             2.65404,2.45224,2.26387,2.10004,1.9629,1.8497,1.74458,1.68712,1.64405,1.63427,1.63981,1.66344,1.72453,1.78653,1.90292,2.02335,2.16903,2.34949,2.53005,2.7524,2.97696,
          2.77571,2.53175,2.31775,2.14106,1.98515,1.85299,1.75364,1.69362,1.63083,1.62149,1.62792,1.67118,1.72536,1.80867,1.92776,2.07206,2.22831,2.42349,2.62538,2.88174,3.12662,
             2.87728,2.6193,2.37427,2.165,2.00334,1.86029,1.75425,1.67991,1.6242,1.60512,1.61927,1.65669,1.72712,1.8246,1.95197,2.1004,2.28778,2.50441,2.75625,3.02319,3.31057,
            2.99413,2.71414,2.43441,2.20486,2.0295,1.86909,1.7652,1.68043,1.62269,1.59653,1.62306,1.66851,1.72861,1.82748,1.98592,2.14914,2.34883,2.58758,2.84584,3.14575,3.47288,
            3.10318,2.77831,2.50831,2.26642,2.06511,1.88857,1.78035,1.6779,1.61583,1.585,1.60781,1.64608,1.74301,1.84645,2.00974,2.19533,2.41724,2.68212,2.96311,3.26512,3.62709,
             3.20224,2.8685,2.55253,2.30502,2.09277,1.92173,1.76325,1.65085,1.59999,1.58892,1.59391,1.65782,1.75363,1.87798,2.03772,2.2207,2.47491,2.7582,3.08242,3.40509,3.78229
             
])
z=np.array([zz[i] for i in range(21*8)])
sigma_z=np.array([math.sqrt(xx[0][i])*zz[i]/1000/math.sqrt(zz[i])/math.sqrt(1.5) for i in range(21*0,21*8)])
initial_guess=[1.8,0,0,0,0,0,0.20,2.2,1]
abc,para=curve_fit(func2,xx,z,sigma=sigma_z,absolute_sigma=True,p0=initial_guess,bounds=bounds,maxfev=150000)
print(abc)
print(para)
z_fit=np.array([func2((xx[0][i],xx[1][i]),*abc) for i in range(21*8)])
chi=0

dev=[]
for i in range(21*8):
    chi+=(z[i]-z_fit[i])**2/sigma_z[i]**2
    dev.append((z[i]-z_fit[i])**2/sigma_z[i]**2)
print(dev)
print(chi)
print(z)
amplitudes=np.linspace(20,48,8)
norm=Normalize(vmin=20,vmax=48)
cmap=plt.get_cmap('viridis')
sm=ScalarMappable(norm=norm,cmap=cmap)
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,4),sharey=True)
for i in range(8):
    ax1.errorbar(xx[1][i*21:i*21+21],z[i*21:i*21+21],fmt='o',yerr=sigma_z[i*21:i*21+21],color=sm.to_rgba(20+i*4),markersize=3,capsize=3,label=r'$L_y=$'+f'{(20+i*4):.0f}')
    x_fit=np.linspace(0.190,0.215,100)
    z_fit=[func2((20+4*i,x_fit[j]),*abc) for j in range(100)]
    ax1.plot(x_fit,z_fit,color=sm.to_rgba(20+i*4))
ax1.set_xlabel(r'$E$',fontdict={'family':'Times New Roman','size':24})
ax1.set_ylabel(r'$\Gamma$',fontdict={'family':'Times New Roman','size':24})
ax1.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
ax1.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax1.set_xticks([0.190, 0.200, 0.210])
ax1.set_yticks([1.5, 2.5, 3.5])
ax1.tick_params(axis='both', which='major', labelsize=16)
norml=[xx[0][i]/xi((xx[0][i],xx[1][i]),*abc) for i in range(168)]
gamma_c=[corrected((xx[0][i],xx[1][i]),*abc) for i in range(168)]
for i in range(8):
    ax2.errorbar(norml[i*21:i*21+21],z[i*21:i*21+21],fmt='o',yerr=sigma_z[i*21:i*21+21],color=sm.to_rgba(20+i*4),markersize=3,capsize=3)
ax2.set_xlabel(r'$L_y / \xi$',fontdict={'family':'Times New Roman','size':24})
ax2.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
ax1.text(0.02, 0.98, '(c)', transform=ax1.transAxes, fontsize=20, va='top')
ax2.text(0.02, 0.98, '(d)', transform=ax2.transAxes, fontsize=20, va='top')
ax2.text(0.7, 0.2, r'$L_z=11$', transform=ax2.transAxes, fontsize=20, va='top',bbox=dict(boxstyle='round', facecolor='white', edgecolor='silver'))
ax2.tick_params(axis='both', which='major', labelsize=16)
plt.tight_layout()
fig.legend(loc='lower left',bbox_to_anchor=(0.175,0.555),fontsize=12,ncol=2)
plt.subplots_adjust(wspace=0)

[ 1.31428180e+00 -2.61576220e+00  3.99773987e+02 -1.96068233e+02
  1.07160465e+02  8.61208334e-01  2.00752443e-01  2.25889363e+00
  2.96335603e-01]
[[ 2.70755651e-02 -2.06905211e-03  2.75680770e-01 -2.88557986e-01
   3.64123230e+00 -5.01418105e-04 -3.29576991e-07  5.21150690e-04
   2.52404389e-02]
 [-2.06905211e-03  2.09876816e-02 -4.27741606e-01  9.31426466e-01
  -6.98360857e+00  7.61830789e-04  5.46633400e-06 -7.62224919e-04
  -1.62367104e-03]
 [ 2.75680770e-01 -4.27741606e-01  4.96227801e+01 -8.29675530e+01
   1.40130723e+03 -1.11450856e-01 -8.30615517e-05  9.59667766e-02
   2.19413782e-01]
 [-2.88557986e-01  9.31426466e-01 -8.29675530e+01  5.92532606e+02
  -8.77006633e+03  1.70353689e-01  3.91558909e-04 -1.82918529e-01
  -2.05147953e-01]
 [ 3.64123230e+00 -6.98360857e+00  1.40130723e+03 -8.77006633e+03
   2.19974266e+05 -3.19230228e+00 -2.98216111e-03  3.64227033e+00
   2.12772338e+00]
 [-5.01418105e-04  7.61830789e-04 -1.11450856e-01  1.70353689e-01
  -3.19230228e+00  5.31166829e-

<Figure size 1000x400 with 2 Axes>